# Express — `res.redirect()`

Redirects the client to a different URL. Express sends a `Location` header plus a redirect status code; the **browser** then makes a second request to that URL.

Default status is **302 (Found)** — a temporary redirect.

```js
res.redirect('/home');
res.redirect(301, '/permanent-home');
```

---

## Redirect targets

### Relative path — same app

```js
res.redirect('/comments');
```

Leading slash = root of the app. Without it (`res.redirect('comments')`) the path resolves relative to the *current* URL, which is almost never what you want. Always lead with `/`.

### Absolute URL — external site

```js
res.redirect('https://www.google.com');
```

### Back to the referring page

```js
// Express 4 only — REMOVED in Express 5
res.redirect('back');
```

> [!warning] Express 5 breaking change
> The magic `'back'` string was removed in Express 5. Do it manually instead:
> ```js
> res.redirect(req.get('Referrer') || '/');
> ```
> The `|| '/'` fallback matters — `Referrer` is absent on direct navigation, and passing `undefined` to `redirect` throws.

---

## Status codes

| Code | Meaning | Method preserved on redirect? | Use when |
|------|---------|-------------------------------|----------|
| `301` | Moved Permanently | No — becomes GET | URL changed for good; tells search engines to update. **Browsers cache this aggressively.** |
| `302` | Found (default) | No — becomes GET | Generic temporary redirect |
| `303` | See Other | No — always GET | Explicitly "go GET this other thing" — the correct code for POST → GET |
| `307` | Temporary Redirect | **Yes** | Temporary, and the POST body must be re-sent |
| `308` | Permanent Redirect | **Yes** | Permanent, and the POST body must be re-sent |

> [!caution] 301 is sticky
> Browsers cache 301s on disk. If you ship a wrong 301 and fix it later, users who already hit it keep getting the old redirect until they clear cache. Use 302 while developing, 301 only when you're certain.

---

## The pattern you'll actually use most: POST → Redirect → GET

After a form submission, **never** render HTML directly from the POST handler. Redirect to a GET route instead.

```js
app.post('/comments', (req, res) => {
    const { username, comment } = req.body;
    comments.push({ username, comment });
    res.redirect('/comments');       // ← not res.render()
});

app.get('/comments', (req, res) => {
    res.render('comments/index', { comments });
});
```

**Why:** if you render from the POST handler, the browser's URL stays on the POST request. Hitting refresh re-submits the form — duplicate comments, duplicate orders, duplicate everything. The redirect leaves the browser sitting on a harmless GET, so refresh is safe.

This is a standard idiom, usually called **PRG (Post/Redirect/Get)**. Nearly every "create" route in a REST-style Express app ends in a redirect.

---

## `res.redirect()` vs `res.render()` vs `res.send()`

| Method | What happens | Browser URL |
|--------|--------------|-------------|
| `res.render('view', data)` | Server builds HTML from a template and sends it | Unchanged |
| `res.send('text')` | Sends a raw response body | Unchanged |
| `res.redirect('/path')` | Sends a `Location` header; browser requests the new URL | **Changes** |

`redirect` costs an extra round trip. `render` doesn't. Use `render` when you're showing the page, `redirect` when you're sending the user somewhere else.

---

## Gotchas

**A redirect ends the response.** Nothing after it will reach the client, and touching `res` again throws `ERR_HTTP_HEADERS_SENT`:

```js
// Broken
app.get('/x', (req, res) => {
    res.redirect('/y');
    res.send('hello');    // ✗ Error: Cannot set headers after they are sent
});

// Fixed — return out
app.get('/x', (req, res) => {
    if (!req.user) return res.redirect('/login');
    res.send('hello');
});
```

The `return` in front of `res.redirect` is the habit worth building.

**Open redirect vulnerability.** Never hand user-controlled input straight to `redirect`:

```js
// ✗ Dangerous — /go?url=https://evil-phishing-site.com
app.get('/go', (req, res) => res.redirect(req.query.url));
```

An attacker sends a link that looks like it points at *your* domain, but bounces the victim to a phishing clone. Validate against an allowlist, or force it to stay internal:

```js
const allowed = ['/home', '/comments', '/profile'];
app.get('/go', (req, res) => {
    const target = req.query.url;
    res.redirect(allowed.includes(target) ? target : '/');
});
```

**Redirects aren't state.** The destination route has no idea a redirect happened. If you need to show "Comment added!" on the next page, you need session-backed flash messages (`connect-flash` or `express-session`), not the redirect itself.

---

## Full example

```js
const express = require('express');
const app = express();

app.use(express.urlencoded({ extended: true }));

app.get('/old-route', (req, res) => {
    res.redirect(301, '/new-route');    // permanent move
});

app.get('/new-route', (req, res) => {
    res.send('Welcome to the new route!');
});

app.listen(3000, () => {
    console.log('LISTENING ON PORT 3000');
});
```

---

## References

- [Express 5 routing guide](https://expressjs.com/en/5x/guide/routing/)
- [Express API — `res.redirect`](https://expressjs.com/en/5x/api.html#res.redirect)
- [MDN — Redirections in HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP/Redirections)